# Strategic Plan -> Top-3 Funding Calls (Phase 1)

This notebook implements the Phase 1 baseline: for each strategic plan, retrieve candidate chunks from Chroma, aggregate at funding-call level, and return a top-3 ranking.


## Setup

Install required packages in the active environment.


In [28]:
%pip install -q chromadb pypdf sentence-transformers pandas tqdm


Note: you may need to restart the kernel to use updated packages.


## Imports

Import libraries for PDF loading, Chroma retrieval, and ranking output.


In [29]:
from pathlib import Path
import re
from typing import Dict, List

import chromadb
import pandas as pd
from chromadb.utils import embedding_functions
from pypdf import PdfReader
from tqdm.auto import tqdm


## Configuration

Centralize paths, collection/model settings, and ranking constants for reproducible runs.


In [30]:
PROJECT_ROOT = Path.cwd()
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"
COLLECTION_NAME = "funding_calls"
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

STRATEGIC_PLAN_DIR = PROJECT_ROOT / "docs" / "strategicplans"
EXPECTED_SP_FILES = {
    "Naples_Federico_Piano_strategico_2021_2026.pdf",
    "Politechnico_di_Milano_2023-2025.pdf",
    "Piano Strategico Luiss 2021-2024 (per sito).pdf",
    "Bocconi_Piano_Strategico2021-2025&Vision2030.pdf",
    "LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf",
    "Sapienza_2021_2027.pdf",
}

N_RESULTS = 30
TOP_K_CALLS = 3
EVIDENCE_PER_CALL = 3
STRONG_HIT_THRESHOLD = 0.55

print(f"Project root: {PROJECT_ROOT}")
print(f"Chroma dir: {CHROMA_DIR}")
print(f"Collection: {COLLECTION_NAME}")
print(f"SP directory: {STRATEGIC_PLAN_DIR}")


Project root: /home/lburmazevic/Projects/EY-AI-Techniques-Solution
Chroma dir: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/data/chroma
Collection: funding_calls
SP directory: /home/lburmazevic/Projects/EY-AI-Techniques-Solution/docs/strategicplans


## Strategic Plans

Load the six strategic plan PDFs and build a clean text representation for each SP query.


In [31]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_sp_text(pdf_path: Path) -> Dict:
    reader = PdfReader(str(pdf_path))
    page_texts: List[str] = []

    for page in reader.pages:
        text = clean_text(page.extract_text() or "")
        if len(text) >= 40:
            page_texts.append(text)

    return {
        "sp_id": pdf_path.stem,
        "source_file": pdf_path.name,
        "text": "\n".join(page_texts),
        "non_empty_pages": len(page_texts),
    }


sp_paths = sorted(STRATEGIC_PLAN_DIR.glob("*.pdf"))
found = {p.name for p in sp_paths}
missing = EXPECTED_SP_FILES - found
if missing:
    raise FileNotFoundError(f"Missing strategic plan files: {sorted(missing)}")

strategic_plans = []
for path in tqdm(sp_paths, desc="Loading strategic plans"):
    if path.name in EXPECTED_SP_FILES:
        strategic_plans.append(extract_sp_text(path))

print(f"Loaded {len(strategic_plans)} strategic plans")
for sp in strategic_plans:
    print(f" - {sp['source_file']}: non_empty_pages={sp['non_empty_pages']}, chars={len(sp['text'])}")


Loading strategic plans: 100%|██████████| 6/6 [00:13<00:00,  2.27s/it]

Loaded 6 strategic plans
 - Bocconi_Piano_Strategico2021-2025&Vision2030.pdf: non_empty_pages=38, chars=85284
 - LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf: non_empty_pages=74, chars=195991
 - Naples_Federico_Piano_strategico_2021_2026.pdf: non_empty_pages=68, chars=68387
 - Piano Strategico Luiss 2021-2024 (per sito).pdf: non_empty_pages=62, chars=55615
 - Politechnico_di_Milano_2023-2025.pdf: non_empty_pages=29, chars=29591
 - Sapienza_2021_2027.pdf: non_empty_pages=31, chars=58569


## Connect to Chroma

Attach to the persistent ChromaDB collection generated during ingestion.


In [32]:
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
collection = client.get_collection(name=COLLECTION_NAME, embedding_function=embed_fn)
print("Chroma collection is ready")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1203.78it/s]


Chroma collection is ready


## Phase 1 Ranking Logic

For each SP, retrieve candidate chunks, aggregate by funding call, compute weighted scores, and keep top-3 calls.


In [33]:
def similarity_from_distance(distance: float) -> float:
    return 1.0 / (1.0 + float(distance))


def rank_calls_for_sp(sp: Dict, n_results: int = N_RESULTS) -> Dict:
    result = collection.query(
        query_texts=[sp['text']],
        n_results=n_results,
        include=['metadatas', 'documents', 'distances'],
    )

    metadatas = result['metadatas'][0]
    documents = result['documents'][0]
    distances = result['distances'][0]

    by_call: Dict[str, Dict] = {}

    for meta, doc, dist in zip(metadatas, documents, distances):
        call = meta.get('source_file', 'unknown')
        page = meta.get('page', None)
        sim = similarity_from_distance(dist)

        if call not in by_call:
            by_call[call] = {
                'call_name': call,
                'similarities': [],
                'pages': set(),
                'evidence': [],
            }

        by_call[call]['similarities'].append(sim)
        if page is not None:
            by_call[call]['pages'].add(page)

        by_call[call]['evidence'].append({
            'source_file': meta.get('source_file', call),
            'source_path': meta.get('source_path', ''),
            'page': page,
            'similarity': round(sim, 4),
            'chunk': doc[:700],
        })

    ranked = []
    for call, payload in by_call.items():
        sims = sorted(payload['similarities'], reverse=True)
        top_sims = sims[:5]

        semantic_score = sum(top_sims) / max(1, len(top_sims))
        coverage_score = min(1.0, len(payload['pages']) / 6.0)
        strong_hits = sum(1 for s in sims if s >= STRONG_HIT_THRESHOLD)
        consistency_score = min(1.0, strong_hits / 5.0)
        final_score = 0.7 * semantic_score + 0.2 * coverage_score + 0.1 * consistency_score

        ranked.append({
            'call_name': call,
            'semantic_score': round(semantic_score, 4),
            'coverage_score': round(coverage_score, 4),
            'consistency_score': round(consistency_score, 4),
            'final_score': round(final_score, 4),
            'evidence': sorted(payload['evidence'], key=lambda x: x['similarity'], reverse=True)[:EVIDENCE_PER_CALL],
        })

    ranked = sorted(ranked, key=lambda x: x['final_score'], reverse=True)

    return {
        'sp_id': sp['sp_id'],
        'sp_file': sp['source_file'],
        'top_calls': ranked[:TOP_K_CALLS],
        'raw_hits': len(metadatas),
    }


## Batch Run

Run the ranking pipeline for all SPs and return a compact table of top recommendations.


In [34]:
phase1_results = []
for sp in tqdm(strategic_plans, desc='Ranking funding calls'):
    phase1_results.append(rank_calls_for_sp(sp))

rows = []
for result in phase1_results:
    for idx, call in enumerate(result['top_calls'], start=1):
        rows.append({
            'sp_file': result['sp_file'],
            'rank': idx,
            'call_name': call['call_name'],
            'final_score': call['final_score'],
            'semantic_score': call['semantic_score'],
            'coverage_score': call['coverage_score'],
            'consistency_score': call['consistency_score'],
        })

ranking_df = pd.DataFrame(rows).sort_values(['sp_file', 'rank']).reset_index(drop=True)
ranking_df


Ranking funding calls: 100%|██████████| 6/6 [00:01<00:00,  3.61it/s]


,sp_file,rank,call_name,final_score,semantic_score,coverage_score,consistency_score
0,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,1,HORIZON-WIDERA-2023-ERA-01-06.pdf,0.6571,0.7101,0.5000,0.6
1,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,2,HORIZON-WIDERA-2022-ERA-01-10.pdf,0.6067,0.7144,0.3333,0.4
2,Bocconi_Piano_Strategico2021-2025&Vision2030.pdf,3,HORIZON-WIDERA-2021-ERA-01-20.pdf,0.6067,0.7143,0.3333,0.4
3,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,1,Partenariati Estesi PNRR.pdf,0.7750,0.6786,1.0000,1.0
4,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,2,PRIN 2022 PNRR.pdf,0.7747,0.6781,1.0000,1.0
5,LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf,3,Ecosistemi dell'Innovazione PNRR.pdf,0.7487,0.6886,0.8333,1.0
6,Naples_Federico_Piano_strategico_2021_2026.pdf,1,PRIN 2022 PNRR.pdf,0.7536,0.6480,1.0000,1.0
7,Naples_Federico_Piano_strategico_2021_2026.pdf,2,Ecosistemi dell'Innovazione PNRR.pdf,0.7241,0.6534,0.8333,1.0
8,Naples_Federico_Piano_strategico_2021_2026.pdf,3,PRIN PNRR 2022 NextGen.pdf,0.7236,0.6527,0.8333,1.0
9,Piano Strategico Luiss 2021-2024 (per sito).pdf,1,Ecosistemi dell'Innovazione PNRR.pdf,0.8192,0.7418,1.0000,1.0


## Evidence Preview

Inspect the top supporting chunks behind each recommended call for quick explainability checks.


In [35]:
def print_evidence_for_sp(sp_file: str):
    matches = [x for x in phase1_results if x['sp_file'] == sp_file]
    if not matches:
        print(f'No result for {sp_file}')
        return

    result = matches[0]
    print(f"Strategic plan: {result['sp_file']}")
    for i, call in enumerate(result['top_calls'], start=1):
        print(f"\nRank {i}: {call['call_name']} | final_score={call['final_score']}")
        for ev in call['evidence']:
            page_label = ev['page'] if ev['page'] is not None else 'n/a'
            print(f" - page={page_label}, sim={ev['similarity']}: {ev['chunk'][:180]}...")

# Example:
print_evidence_for_sp(phase1_results[0]['sp_file'])


Strategic plan: Bocconi_Piano_Strategico2021-2025&Vision2030.pdf

Rank 1: HORIZON-WIDERA-2023-ERA-01-06.pdf | final_score=0.6571
 - page=4, sim=0.7108: Horizon Europe - Work Programme 2023-2025 Widening participation and strengthening the European Research Area Part 11 - Page 103 of 193 faces in its capacity to better understand c...
 - page=3, sim=0.7098: Horizon Europe - Work Programme 2023-2025 Widening participation and strengthening the European Research Area Part 11 - Page 102 of 193 • Organise the material to be uploaded on th...
 - page=1, sim=0.7097: Horizon Europe - Work Programme 2023-2025 Widening participation and strengthening the European Research Area Part 11 - Page 100 of 193 support mutual learning and exchange of good...

Rank 2: HORIZON-WIDERA-2022-ERA-01-10.pdf | final_score=0.6067
 - page=2, sim=0.7172: Horizon Europe - Work Programme 2021-2022 Widening participation and strengthening the European Research Area Part 11 - Page 107 of 176  Identification of common 

## Phase 2 XAI Configuration

Define explainability settings and theme categories used to generate auditable rationale.


In [ ]:
XAI_TOP_THEMES = 4
XAI_EVIDENCE_PER_CALL = 3
XAI_MAX_GAPS = 4

GAP_MIN_CALL_SIGNAL = 0.2
GAP_MAX_SP_SIGNAL = 0.1


## Theme Lexicon

Map broad strategic themes to multilingual keywords for transparent, deterministic theme matching.


In [ ]:
THEME_LEXICON = {
    "sustainability_climate": [
        "sustainability", "sustainable", "climate", "green", "decarbonization",
        "circular economy", "energy transition", "biodiversity", "sdg",
        "sostenibilita", "sostenibile", "clima", "transizione energetica",
        "economia circolare", "decarbonizzazione", "biodiversita"
    ],
    "ai_data_digital": [
        "artificial intelligence", "ai", "machine learning", "data", "digital",
        "digitalization", "digitization", "big data", "cloud", "cybersecurity",
        "intelligenza artificiale", "apprendimento automatico", "dati", "digitale",
        "digitalizzazione", "transizione digitale"
    ],
    "internationalization_collaboration": [
        "international", "internationalization", "cross-border", "european", "consortium",
        "partnership", "collaboration", "mobility", "network", "joint",
        "internazionale", "internazionalizzazione", "europeo", "consorzio",
        "partenariato", "collaborazione", "mobilita", "rete", "congiunto"
    ],
    "innovation_transfer_industry": [
        "innovation", "technology transfer", "valorization", "startup", "spin-off",
        "industry", "industrial", "commercialization", "patent",
        "innovazione", "trasferimento tecnologico", "valorizzazione",
        "industria", "industriale", "brevett"
    ],
    "skills_education_capacity": [
        "skills", "competence", "training", "education", "lifelong learning",
        "upskilling", "reskilling", "capacity building", "doctoral", "curriculum",
        "competenze", "formazione", "istruzione", "apprendimento permanente", "dottorato"
    ],
    "inclusion_gender_social": [
        "inclusion", "equity", "equality", "gender", "diversity", "accessibility",
        "social impact", "vulnerable", "cohesion",
        "inclusione", "equita", "uguaglianza", "genere", "diversita",
        "accessibilita", "impatto sociale", "vulnerabili", "coesione"
    ],
    "research_infrastructure_excellence": [
        "research infrastructure", "infrastructure", "laboratory", "equipment",
        "excellence", "scientific excellence", "facility", "platform",
        "infrastruttura di ricerca", "infrastruttura", "laboratorio", "attrezzature",
        "eccellenza", "eccellenza scientifica", "piattaforma"
    ],
    "governance_policy_reform": [
        "governance", "reform", "policy", "regulation", "institutional",
        "coordination", "roadmap", "implementation", "monitoring", "evaluation",
        "riforma", "politica", "regolazione", "istituzionale",
        "coordinamento", "attuazione", "monitoraggio", "valutazione"
    ],
}


## XAI Builders

Convert evidence chunks into matched themes and actionable gaps for each recommended funding call.


In [ ]:
def compute_theme_signal_from_text(text: str) -> Dict[str, Dict]:
    normalized = clean_text(text).lower()
    raw_hits = {}
    total_hits = 0

    for theme, keywords in THEME_LEXICON.items():
        hits = 0
        matched = []
        for keyword in keywords:
            count = normalized.count(keyword.lower())
            if count > 0:
                hits += count
                matched.append(keyword)
        raw_hits[theme] = {"hits": hits, "matched_keywords": matched}
        total_hits += hits

    signal = {}
    for theme, payload in raw_hits.items():
        score = (payload["hits"] / total_hits) if total_hits > 0 else 0.0
        signal[theme] = {
            "hits": payload["hits"],
            "score": round(score, 4),
            "matched_keywords": payload["matched_keywords"][:8],
        }

    return signal


def derive_matched_themes(sp_signal: Dict, call_signal: Dict, top_n: int = XAI_TOP_THEMES) -> List[Dict]:
    rows = []
    for theme in THEME_LEXICON.keys():
        sp_score = sp_signal.get(theme, {}).get("score", 0.0)
        call_score = call_signal.get(theme, {}).get("score", 0.0)
        rows.append({
            "theme": theme,
            "sp_score": round(sp_score, 4),
            "call_score": round(call_score, 4),
            "match_strength": round(min(sp_score, call_score), 4),
            "keywords_in_call": call_signal.get(theme, {}).get("matched_keywords", [])[:5],
        })

    rows.sort(key=lambda x: (x["match_strength"], x["call_score"]), reverse=True)
    return rows[:top_n]


def derive_key_gaps(sp_signal: Dict, call_signal: Dict, max_gaps: int = XAI_MAX_GAPS) -> List[Dict]:
    action_map = {
        "sustainability_climate": "SP should strengthen measurable sustainability and climate actions with clear KPIs.",
        "ai_data_digital": "SP should strengthen AI/data/digital transition actions with concrete implementation milestones.",
        "internationalization_collaboration": "SP should strengthen international collaboration targets (consortia, mobility, partnerships).",
        "innovation_transfer_industry": "SP should strengthen technology transfer and industry engagement pathways.",
        "skills_education_capacity": "SP should strengthen skills and capacity-building plans with measurable outcomes.",
        "inclusion_gender_social": "SP should strengthen inclusion and gender/social impact commitments with indicators.",
        "research_infrastructure_excellence": "SP should strengthen research infrastructure and excellence positioning.",
        "governance_policy_reform": "SP should strengthen governance and implementation roadmap clarity.",
    }

    candidates = []
    for theme in THEME_LEXICON.keys():
        sp_score = sp_signal.get(theme, {}).get("score", 0.0)
        call_score = call_signal.get(theme, {}).get("score", 0.0)
        if call_score >= GAP_MIN_CALL_SIGNAL and sp_score <= GAP_MAX_SP_SIGNAL:
            candidates.append((theme, call_score - sp_score, sp_score, call_score))

    candidates.sort(key=lambda x: x[1], reverse=True)

    gaps = []
    for theme, gap_strength, sp_score, call_score in candidates[:max_gaps]:
        gaps.append({
            "theme": theme,
            "gap_strength": round(gap_strength, 4),
            "sp_score": round(sp_score, 4),
            "call_score": round(call_score, 4),
            "action": action_map[theme],
        })

    return gaps


def build_call_xai(sp_text: str, call_row: Dict) -> Dict:
    evidence = call_row.get("evidence", [])[:XAI_EVIDENCE_PER_CALL]
    evidence_text = " ".join([item.get("chunk", "") for item in evidence])

    sp_signal = compute_theme_signal_from_text(sp_text)
    call_signal = compute_theme_signal_from_text(evidence_text)

    supporting_chunks = []
    for item in evidence:
        supporting_chunks.append({
            "source_file": item.get("source_file", call_row.get("call_name")),
            "source_path": item.get("source_path", ""),
            "page": item.get("page"),
            "similarity": item.get("similarity"),
            "excerpt": item.get("chunk", "")[:320],
        })

    return {
        "call_name": call_row.get("call_name"),
        "final_score": call_row.get("final_score"),
        "score_breakdown": {
            "semantic_score": call_row.get("semantic_score"),
            "coverage_score": call_row.get("coverage_score"),
            "consistency_score": call_row.get("consistency_score"),
        },
        "supporting_chunks": supporting_chunks,
        "matched_themes": derive_matched_themes(sp_signal, call_signal),
        "key_gaps": derive_key_gaps(sp_signal, call_signal),
    }


## Phase 2 Build

Attach explainability objects to each top-3 recommendation for every strategic plan.


In [ ]:
phase2_results = []

for result in phase1_results:
    sp = next((item for item in strategic_plans if item["source_file"] == result["sp_file"]), None)
    if sp is None:
        continue

    calls_with_xai = []
    for call in result["top_calls"]:
        call_copy = dict(call)
        call_copy["xai"] = build_call_xai(sp["text"], call_copy)
        calls_with_xai.append(call_copy)

    phase2_results.append({
        "sp_id": result["sp_id"],
        "sp_file": result["sp_file"],
        "raw_hits": result["raw_hits"],
        "top_calls": calls_with_xai,
    })

print(f"Built Phase 2 XAI objects for {len(phase2_results)} strategic plans")


## Phase 2 Summary Table

Show top-3 calls with matched themes and improvement actions in one compact output table.


In [ ]:
xai_rows = []

for result in phase2_results:
    for rank, call in enumerate(result["top_calls"], start=1):
        xai = call["xai"]
        themes = ", ".join([item["theme"] for item in xai["matched_themes"]])
        actions = " | ".join([item["action"] for item in xai["key_gaps"]])

        xai_rows.append({
            "sp_file": result["sp_file"],
            "rank": rank,
            "call_name": xai["call_name"],
            "final_score": xai["final_score"],
            "matched_themes": themes,
            "improvement_actions": actions,
        })

xai_df = pd.DataFrame(xai_rows).sort_values(["sp_file", "rank"]).reset_index(drop=True)
xai_df


## Phase 2 Evidence Preview

Inspect full evidence, matched themes, and gaps for one strategic plan recommendation set.


In [ ]:
def print_xai_for_sp(sp_file: str) -> None:
    matches = [row for row in phase2_results if row["sp_file"] == sp_file]
    if not matches:
        print(f"No Phase 2 result found for {sp_file}")
        return

    result = matches[0]
    print(f"Strategic plan: {result["sp_file"]}")

    for idx, call in enumerate(result["top_calls"], start=1):
        xai = call["xai"]
        print(f"\nRank {idx}: {xai["call_name"]} | final_score={xai["final_score"]}")
        print(f" Score breakdown: semantic={xai["score_breakdown"]["semantic_score"]}, coverage={xai["score_breakdown"]["coverage_score"]}, consistency={xai["score_breakdown"]["consistency_score"]}")

        print(" Matched themes:")
        for theme in xai["matched_themes"]:
            print(f"  - {theme["theme"]} (match={theme["match_strength"]}, call={theme["call_score"]}, sp={theme["sp_score"]})")

        print(" Key gaps:")
        if not xai["key_gaps"]:
            print("  - No critical gaps detected from current evidence.")
        else:
            for gap in xai["key_gaps"]:
                print(f"  - {gap["action"]} (gap={gap["gap_strength"]})")

        print(" Supporting chunks:")
        for ev in xai["supporting_chunks"]:
            page = ev["page"] if ev["page"] is not None else "n/a"
            print(f"  - source={ev["source_file"]}, page={page}, sim={ev["similarity"]}")
            print(f"    {ev["excerpt"][:220]}...")


print_xai_for_sp(phase2_results[0]["sp_file"])
